<a href="https://colab.research.google.com/github/Divyansh-yecho/Project_divyansh_yecho/blob/main/Project_Run_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
masoudnickparvar_brain_tumor_mri_dataset_path = kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

for dirname, dirs, files in os.walk('/kaggle/input'):
    for f in files[:2]:
        print(os.path.join(dirname, f))
    if dirname != '/kaggle/input':
        break

In [ ]:
print(os.listdir('/kaggle/input/datasets/masoudnickparvar'))

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score

BASE_DIR  = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/'
TRAIN_DIR = BASE_DIR + 'Training/'
TEST_DIR  = BASE_DIR + 'Testing/'

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASS_NAMES)}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
print("=== TRAINING SET ===")
total_train = 0
for cls in CLASS_NAMES:
    count = len(os.listdir(os.path.join(TRAIN_DIR, cls)))
    total_train += count
    print(f"  {cls}: {count} images")
print(f"  Total: {total_train}")

print("\n=== TESTING SET ===")
total_test = 0
for cls in CLASS_NAMES:
    count = len(os.listdir(os.path.join(TEST_DIR, cls)))
    total_test += count
    print(f"  {cls}: {count} images")
print(f"  Total: {total_test}")
print(f"\nGrand Total: {total_train + total_test} images")

In [ ]:
filepaths, labels = [], []
for cls in CLASS_NAMES:
    folder = os.path.join(TRAIN_DIR, cls)
    for fname in os.listdir(folder):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            filepaths.append(os.path.join(folder, fname))
            labels.append(CLASS_TO_IDX[cls])

train_df = pd.DataFrame({'filepath': filepaths, 'label': labels})

test_fps, test_lbs = [], []
for cls in CLASS_NAMES:
    folder = os.path.join(TEST_DIR, cls)
    for fname in os.listdir(folder):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            test_fps.append(os.path.join(folder, fname))
            test_lbs.append(CLASS_TO_IDX[cls])

test_df = pd.DataFrame({'filepath': test_fps, 'label': test_lbs})

train_df, val_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df['label'], random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['filepath']).convert('RGB')
        label = self.df.iloc[idx]['label']
        if self.transform:
            img = self.transform(img)
        return img, label


train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = BrainTumorDataset(train_df, transform=train_transforms)
val_dataset   = BrainTumorDataset(val_df,   transform=eval_transforms)
test_dataset  = BrainTumorDataset(test_df,  transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 4)
)

model = model.to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
print("Ready!")

In [ ]:
def run_epoch(model, loader, optimizer, criterion, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss, correct = 0, 0
    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)


EPOCHS = 25
train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = 0.0

print("Starting training...\n")
for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer, criterion, training=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   optimizer, criterion, training=False)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    current_lr = optimizer.param_groups[0]['lr']
    saved = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        saved = "  [saved]"

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS}  "
        f"train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  "
        f"val_loss={val_loss:.4f}  val_acc={val_acc:.3f}  "
        f"lr={current_lr:.2e}{saved}"
    )

print(f"\nBest Val Accuracy: {best_val_acc:.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_accs, label='Train Accuracy', color='blue')
ax1.plot(val_accs,   label='Val Accuracy',   color='orange')
ax1.set_title('Accuracy over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(train_losses, label='Train Loss', color='blue')
ax2.plot(val_losses,   label='Val Loss',   color='orange')
ax2.set_title('Loss over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        outputs = model(imgs.to(device))
        all_preds  += outputs.argmax(1).cpu().tolist()
        all_labels += labels.tolist()

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"Test Accuracy: {test_acc:.3f} ({test_acc*100:.1f}%)\n")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
axes = axes.flatten()

sample = test_df.sample(16, random_state=42).reset_index(drop=True)

for i, row in sample.iterrows():
    img_orig   = Image.open(row['filepath']).convert('RGB')
    img_tensor = eval_transforms(img_orig).unsqueeze(0).to(device)

    with torch.no_grad():
        output     = model(img_tensor)
        pred       = output.argmax(1).item()
        confidence = torch.softmax(output, dim=1).max().item()

    true_label = CLASS_NAMES[row['label']]
    pred_label = CLASS_NAMES[pred]
    color = 'green' if pred_label == true_label else 'red'

    axes[i].imshow(img_orig, cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(
        f"True: {true_label}\nPred: {pred_label}\n{confidence*100:.1f}%",
        color=color, fontsize=9
    )

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150)
plt.show()

In [ ]:
wrong_idxs = [i for i, (p, l) in enumerate(zip(all_preds, all_labels)) if p != l]
print(f"Total misclassified: {len(wrong_idxs)} / {len(all_labels)}")

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
axes = axes.flatten()

for plot_i, idx in enumerate(wrong_idxs[:16]):
    row        = test_df.iloc[idx]
    img_orig   = Image.open(row['filepath']).convert('RGB')
    img_tensor = eval_transforms(img_orig).unsqueeze(0).to(device)

    with torch.no_grad():
        output     = model(img_tensor)
        probs      = torch.softmax(output, dim=1)[0]
        pred       = probs.argmax().item()
        confidence = probs[pred].item()

    axes[plot_i].imshow(img_orig, cmap='gray')
    axes[plot_i].axis('off')
    axes[plot_i].set_title(
        f"True: {CLASS_NAMES[row['label']]}\nPred: {CLASS_NAMES[pred]}\n{confidence*100:.1f}%",
        color='red', fontsize=9
    )

for j in range(len(wrong_idxs[:16]), 16):
    axes[j].axis('off')

plt.suptitle('Misclassified Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('misclassified_samples.png', dpi=150)
plt.show()

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)

colors = ['#2ecc71' if acc >= 0.90 else '#e67e22' if acc >= 0.75 else '#e74c3c'
          for acc in per_class_acc]

plt.figure(figsize=(10, 6))
bars = plt.bar(CLASS_NAMES, per_class_acc * 100, color=colors, edgecolor='black', width=0.5)

for bar, acc in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc*100:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.title('Per-Class Accuracy', fontsize=15, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 110)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150)
plt.show()

In [ ]:
model.eval()
class_confidences = {cls: [] for cls in CLASS_NAMES}

with torch.no_grad():
    for imgs, labels in test_loader:
        outputs = model(imgs.to(device))
        probs   = torch.softmax(outputs, dim=1)
        for j in range(len(labels)):
            true_cls = CLASS_NAMES[labels[j].item()]
            class_confidences[true_cls].append(probs[j].max().item() * 100)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, cls in enumerate(CLASS_NAMES):
    axes[i].hist(class_confidences[cls], bins=20, color='steelblue', edgecolor='black', alpha=0.8)
    axes[i].set_title(f'{cls} — Avg: {np.mean(class_confidences[cls]):.1f}%',
                      fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Confidence (%)')
    axes[i].set_ylabel('Count')
    axes[i].axvline(np.mean(class_confidences[cls]), color='red', linestyle='--', linewidth=2)
    axes[i].grid(alpha=0.3)

plt.suptitle('Prediction Confidence Distribution by True Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=150)
plt.show()

In [ ]:
precision = precision_score(all_labels, all_preds, average='weighted')
recall    = recall_score(all_labels, all_preds, average='weighted')
f1        = f1_score(all_labels, all_preds, average='weighted')

fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')

metrics = [
    ['Metric', 'Score'],
    ['Test Accuracy',      f'{test_acc*100:.2f}%'],
    ['Best Val Accuracy',  f'{best_val_acc*100:.2f}%'],
    ['Weighted Precision', f'{precision*100:.2f}%'],
    ['Weighted Recall',    f'{recall*100:.2f}%'],
    ['Weighted F1 Score',  f'{f1*100:.2f}%'],
    ['Total Test Images',  str(len(test_df))],
    ['Classes',            '4 (Glioma, Meningioma, No Tumor, Pituitary)'],
    ['Model',              'ResNet50 + Transfer Learning'],
]

table = ax.table(cellText=metrics[1:], colLabels=metrics[0], loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(13)
table.scale(1.5, 2.2)

for j in range(2):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

for i in range(1, len(metrics)):
    for j in range(2):
        table[i, j].set_facecolor('#ecf0f1' if i % 2 == 0 else 'white')

plt.title('Brain Tumor MRI Classification — Final Results',
          fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('final_summary.png', dpi=150)
plt.show()

print(f"\nTest Accuracy:      {test_acc*100:.2f}%")
print(f"Weighted Precision: {precision*100:.2f}%")
print(f"Weighted Recall:    {recall*100:.2f}%")
print(f"Weighted F1 Score:  {f1*100:.2f}%")

In [ ]:
import os
import shutil

TEST_DIR = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing/'
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']

os.makedirs('data', exist_ok=True)

for cls in CLASS_NAMES:
    folder = os.path.join(TEST_DIR, cls)
    images = sorted(os.listdir(folder))[:10]
    for i, fname in enumerate(images, 1):
        src = os.path.join(folder, fname)
        dst = os.path.join('data', f"{cls}_{i:02d}.jpg")
        shutil.copy(src, dst)
        print(f"Copied: {dst}")

print(f"\nTotal: {len(os.listdir('data'))} images")

In [ ]:
import os
import shutil

TEST_DIR = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing/'
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']

os.makedirs('/kaggle/working/data', exist_ok=True)

for cls in CLASS_NAMES:
    folder = os.path.join(TEST_DIR, cls)
    images = sorted(os.listdir(folder))[:10]
    for i, fname in enumerate(images, 1):
        src = os.path.join(folder, fname)
        dst = f'/kaggle/working/data/{cls}_{i:02d}.jpg'
        shutil.copy(src, dst)
        print(f"Copied: {dst}")

print(f"\nTotal: {len(os.listdir('/kaggle/working/data'))} images")

In [ ]:
from kaggle_secrets import UserSecretsClient
import zipfile
import os

with zipfile.ZipFile('/kaggle/working/data.zip', 'w') as zipf:
    for fname in os.listdir('/kaggle/working/data'):
        zipf.write(f'/kaggle/working/data/{fname}', fname)

print("data.zip created")

In [ ]:
import zipfile
import os

with zipfile.ZipFile('/kaggle/working/data.zip', 'w') as zipf:
    for fname in os.listdir('/kaggle/working/data'):
        zipf.write(f'/kaggle/working/data/{fname}', fname)

print("Done:", len(os.listdir('/kaggle/working/data')), "files zipped")

In [ ]:
import torch
weights = torch.load('best_model.pth', map_location='cpu')
print(type(weights))
print(f"Number of layers: {len(weights)}")
for k, v in list(weights.items())[:5]:
    print(k, v.shape)

<class 'collections.OrderedDict'>
Number of layers: 322
conv1.weight torch.Size([64, 3, 7, 7])
bn1.weight torch.Size([64])
bn1.bias torch.Size([64])
bn1.running_mean torch.Size([64])
bn1.running_var torch.Size([64])
